# Churn Prediction – Logistic Regression

This notebook builds a churn label from RFM and trains a **Logistic Regression** model.

**Steps**: Load CSV → Parse dates → Invoice aggregation → RFM → Churn labeling (90th percentile gap) → Features → Train/Test → Train Logistic Regression → Evaluate → Save artifacts.

> Edit the CONFIG to point to your CSV.


In [ ]:
# --- CONFIG ---
INPUT_CSV = "output_Sarika.csv"
DATE_COL = "date"
INVOICE_NO_COL = "invoice_no"
RETAILER_CODE_COL = "retailer_code"
RETAILER_NAME_COL = "retailer_name"
INVOICE_VALUE_COL = "invoice_value"
TEST_SIZE = 0.25
RANDOM_STATE = 42
MIN_CHURN_DAYS = 60
MAX_CHURN_DAYS = 180
# -------------


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import joblib

# Load
df = pd.read_csv(INPUT_CSV)

# Dates
def parse_dates(s):
    s = s.astype(str)
    d1 = pd.to_datetime(s, errors="coerce", dayfirst=True)
    d2 = pd.to_datetime(s, errors="coerce", dayfirst=False)
    return d1.fillna(d2)

df["date_parsed"] = parse_dates(df[DATE_COL])
df[INVOICE_VALUE_COL] = pd.to_numeric(df[INVOICE_VALUE_COL], errors="coerce").fillna(0.0)

# Invoice aggregation
invoice_agg = (
    df.groupby([INVOICE_NO_COL, RETAILER_CODE_COL, RETAILER_NAME_COL], as_index=False)
      .agg(sum_invoice_value=(INVOICE_VALUE_COL, "sum"),
           invoice_date=("date_parsed", "max"))
)

# RFM
ref_date = invoice_agg["invoice_date"].max() + pd.Timedelta(days=1)
rfm = (
    invoice_agg.groupby([RETAILER_CODE_COL, RETAILER_NAME_COL], as_index=False)
      .agg(First_Purchase_Date=("invoice_date","min"),
           Last_Purchase_Date=("invoice_date","max"),
           Frequency=(INVOICE_NO_COL,"nunique"),
           Monetary=("sum_invoice_value","sum"))
)
rfm["Recency"] = (ref_date - rfm["Last_Purchase_Date"]).dt.days

# Churn threshold via interpurchase gaps
invoice_agg_sorted = invoice_agg.sort_values([RETAILER_CODE_COL, "invoice_date"])
gaps = []
for rid, grp in invoice_agg_sorted.groupby(RETAILER_CODE_COL):
    dates = grp["invoice_date"].dropna().values
    if len(dates) >= 2:
        diffs = np.diff(dates).astype("timedelta64[D]").astype(int)
        gaps.extend(diffs.tolist())

if len(gaps) > 0:
    q90 = float(np.percentile(gaps, 90))
    churn_threshold_days = int(np.clip(q90, MIN_CHURN_DAYS, MAX_CHURN_DAYS))
else:
    churn_threshold_days = 90

rfm["churn"] = (rfm["Recency"] > churn_threshold_days).astype(int)

# Features
feat = rfm.copy()
feat["TenureDays"] = (feat["Last_Purchase_Date"] - feat["First_Purchase_Date"]).dt.days
feat["AOV"] = feat["Monetary"] / feat["Frequency"]

# Basic gap stats
gap_stats = []
for rid, grp in invoice_agg_sorted.groupby(RETAILER_CODE_COL):
    dates = grp["invoice_date"].dropna().values
    if len(dates) >= 2:
        diffs = np.diff(dates).astype("timedelta64[D]").astype(int)
        gap_stats.append({"retailer_code": rid, "avg_gap": float(np.mean(diffs)), "max_gap": float(np.max(diffs)),
                          "std_gap": float(np.std(diffs)), "n_gaps": int(len(diffs))})
    else:
        gap_stats.append({"retailer_code": rid, "avg_gap": 0.0, "max_gap": 0.0, "std_gap": 0.0, "n_gaps": 0})

gap_df = pd.DataFrame(gap_stats)
feat = feat.merge(gap_df, left_on=RETAILER_CODE_COL, right_on="retailer_code", how="left")

feature_cols = ["Recency","Frequency","Monetary","TenureDays","AOV","avg_gap","max_gap","std_gap","n_gaps"]
feat[feature_cols] = feat[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

X = feat[feature_cols]
y = feat["churn"]

# Split
if y.nunique() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
else:
    X_train, X_test, y_train, y_test = X, X, y, y

print(f"Reference: {ref_date.date()} | Churn threshold (days): {churn_threshold_days}")
print("Target distribution:", y.value_counts().to_dict())

def collect_metrics(name, y_true, y_pred, y_prob):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_true.nunique()>1 else np.nan
    }


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:,1]

metrics = [collect_metrics("LogisticRegression", y_test, y_pred, y_prob)]
import pandas as pd
metrics_df = pd.DataFrame(metrics)
metrics_df
# Save
out = Path(".")
metrics_df.to_csv(out / "logreg_churn_metrics.csv", index=False)
pd.DataFrame({"retailer_code": feat.loc[X_test.index,"retailer_code"].values,
              "true": y_test.values, "pred": y_pred, "prob": y_prob}).to_csv(out / "logreg_churn_predictions.csv", index=False)
joblib.dump(pipe, out / "logreg_churn_model.pkl")

# Report
with PdfPages(out / "logreg_churn_report.pdf") as pdf:
    if y_test.nunique()>1:
        fpr,tpr,_=roc_curve(y_test,y_prob); plt.figure(figsize=(7,6))
        plt.plot(fpr,tpr,label="LogisticRegression"); plt.plot([0,1],[0,1],'--')
        plt.title("ROC - Logistic Regression"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.tight_layout(); pdf.savefig(); plt.close()
    cm = confusion_matrix(y_test,y_pred); plt.figure(figsize=(5,4)); plt.imshow(cm,aspect="auto")
    plt.title("Confusion Matrix - Logistic Regression"); plt.xlabel("Pred"); plt.ylabel("True")
    for (i,j),v in np.ndenumerate(cm): plt.text(j,i,int(v),ha="center",va="center")
    plt.tight_layout(); pdf.savefig(); plt.close()
